In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonTR(model=model, tr_method="dogleg", solver="cg")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.22138971090316772
epoch:  1, loss: 0.03039557859301567
epoch:  2, loss: 0.03012710064649582
epoch:  3, loss: 0.040744658559560776
epoch:  4, loss: 0.03163963556289673
epoch:  5, loss: 0.035395294427871704
epoch:  6, loss: 0.035395294427871704
epoch:  7, loss: 0.03447197377681732
epoch:  8, loss: 0.031545720994472504
epoch:  9, loss: 0.030895400792360306
epoch:  10, loss: 0.03019370511174202
epoch:  11, loss: 0.02880275435745716
epoch:  12, loss: 0.03046596609055996
epoch:  13, loss: 0.029550405219197273
epoch:  14, loss: 0.029331207275390625
epoch:  15, loss: 0.029161233454942703
epoch:  16, loss: 0.028263594955205917
epoch:  17, loss: 0.028263594955205917
epoch:  18, loss: 0.02816751040518284
epoch:  19, loss: 0.028181008994579315
epoch:  20, loss: 0.028423437848687172
epoch:  21, loss: 0.028071098029613495
epoch:  22, loss: 0.028255103155970573
epoch:  23, loss: 0.02799295261502266
epoch:  24, loss: 0.02812405303120613
epoch:  25, loss: 0.028351526707410812
epoch: 

KeyboardInterrupt: 

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -4211.061226519849
Test metrics:  R2 = -3753.3324691155976
